In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

In [4]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [7]:
question = 'What course am i on now?'
answer = llm(question)
print(answer)

I can’t see your current course or account details from here.

If you want, I can help you figure it out if you tell me:
- the platform or school name,
- the app/site you’re using,
- or paste a screenshot/text from your dashboard.

If you meant a navigation or map “course,” tell me what context you’re in.


In [10]:
context = '''
I just discovered the course. Can I still join?

Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.
edit on GitHub
#
Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?

You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.
edit on GitHub
#
What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?

The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram and Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Don’t post questions in chat as they may be missed if the room is very active.
edit on GitHub
#
How should I start the course and follow the weekly workflow?

Start with the LLM Zoomcamp docs, the general Zoomcamp logistics docs, and the LLM Zoomcamp GitHub repository.

You can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the course management platform.

A typical workflow is:

    Watch the lesson videos.
    Work through the lesson notebooks/code.
    Read the homework instructions on GitHub.
    Submit answers through the course platform before the deadline.

Homework is similar to the lesson flow, but uses a different dataset or slightly different task.
edit on GitHub
#
Leaderboard: I am not on the leaderboard / how do I know which one I am on the leaderboard?

When you set up your account, you are automatically assigned a random name, such as “Lucid Elbakyan.” Click on the "Jump to your record on the leaderboard" link to find your entry.

If you want to see what your Display name is, click on the "Edit Course Profile" button.

image #1

    First field: This is your nickname/displayed name. You can change it if you want to be known by your Slack username, GitHub username, or any other nickname of your choice. This is useful if you want to remain anonymous.
    Second field: Change this to your official name as in your identification documents—passport, national ID card, driver's license, etc. This is mandatory if you do not want "Lucid Elbakyan" on your certificate. This name will appear on your Certificate!

edit on GitHub
#
Certificate: Can I follow the course in a self-paced mode and get a certificate?

No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.
edit on GitHub
#
I missed the first homework - can I still get a certificate?

Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.
edit on GitHub
#
Homework: Why does the content keep changing?

If the homework title contains [DRAFT], it means the homework is not ready yet.

The homework is ready only when both are true:

    The homework form is open on the course management platform.
    The homework title does not contain [DRAFT].

Until then, the content can still change. Working on the material or homework in advance is at your own risk, because the final version can be different.
edit on GitHub
#
When will the course be offered next?

Summer 2027.

'''

In [11]:
prompt = f'''
Your task is to answer questions from the course participants based on the provided context. 

Use the context to find relevant information and provide accurate answers. If the answer is not found in the context, reply "I don't know". 

Question:
{question}

Context:
{context}
'''

In [12]:
print(prompt)


Your task is to answer questions from the course participants based on the provided context. 

Use the context to find relevant information and provide accurate answers. If the answer is not found in the context, reply "I don't know". 

Question:
What course am i on now?

Context:

I just discovered the course. Can I still join?

Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.
edit on GitHub
#
Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?

You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.
edit on GitHub
#
What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?

The zoom link is only published to instructors/presenters/TAs.

Studen

In [28]:
question = 'When does the course start?'
answer = llm(prompt)
print(answer)

Summer 2027.


In [ ]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)



In [21]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()


In [23]:
documents = []
url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
    course_url = f'{url_prefix}{course['path']}'

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)


1346

In [25]:
from minsearch import Index

In [26]:
index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)

index.fit(documents)

In [34]:
def search(question, course='llm-zoomcamp'):
    boost_dict={'question': 2.0, 'section': 0.5}
    filter_dict={'course': course}

    return index.search(question, 
                        filter_dict=filter_dict,
                        num_results=5,
                        boost_dict=boost_dict
                        )
   

In [51]:
question = 'How do you submit homework?'
search_results = search('How do you submit homework?')
[doc['question'] for doc in search_results]

['Do homework answers need to match the options exactly?',
 'How do I get token counts for Module 1 homework if I use a different provider?',
 'What should I do if homework questions feel unclear?',
 'Where can I find the homework questions?',
 'Do we submit 2 projects, what does attempt 1 and 2 mean?']

In [61]:
INSTRUCTIONS = '''Your task is to answer questions from the course participants based on the provided context. 

Use the context to find relevant information and provide accurate answers. If the answer is not found in the context, reply "I don't know".'''

In [46]:
USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}
'''

In [41]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('')

    return '\n'.join(lines).strip()

In [44]:
context = build_context(search_results)
print(context)

Module 1 Homework
Q: Do homework answers need to match the options exactly?
A: Not always. If your numeric answer is close to one of the options, choose the closest option.

Small differences can come from:

- Slightly different filtering.
- Different dataset versions.
- Floating-point or rounding differences.
- Different model/provider behavior.

If your answer is far from every option, re-check the question, the dataset version, and the GitHub homework instructions.

See the general homework guidance: [homework logistics](https://datatalks.club/docs/courses/zoomcamp-logistics/homework/).

Module 1 Homework
Q: How do I get token counts for Module 1 homework if I use a different provider?
A: For the current Module 1 homework, get the token count from the model response object.

For example, OpenAI-compatible clients usually return usage information on the response, such as `response.usage.input_tokens` or `response.usage.prompt_tokens`, depending on the API style.

If you use a non-Ope

In [50]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(question=question, context=context)
    return prompt.strip()


In [53]:
prompt = build_prompt(question, search_results)
print(prompt)

Question:
How do you submit homework?

Context:
Module 1 Homework
Q: Do homework answers need to match the options exactly?
A: Not always. If your numeric answer is close to one of the options, choose the closest option.

Small differences can come from:

- Slightly different filtering.
- Different dataset versions.
- Floating-point or rounding differences.
- Different model/provider behavior.

If your answer is far from every option, re-check the question, the dataset version, and the GitHub homework instructions.

See the general homework guidance: [homework logistics](https://datatalks.club/docs/courses/zoomcamp-logistics/homework/).

Module 1 Homework
Q: How do I get token counts for Module 1 homework if I use a different provider?
A: For the current Module 1 homework, get the token count from the model response object.

For example, OpenAI-compatible clients usually return usage information on the response, such as `response.usage.input_tokens` or `response.usage.prompt_tokens`, d

In [55]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

print(response.output_text)

Submit it through the **course platform**—that’s where homework submissions are handled.

If you mean the **Module 1 homework**, first make sure you’ve followed the instructions in the **GitHub homework page**:
- https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/01-agentic-rag/homework.md

Then use the **course management platform** to enter your answers before the deadline.

If you want, I can also point you to the exact submission page for your cohort.


In [56]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_009f373e38174c09006a33a0212c7c81a3b7b6d0ede4d8ac71",
  "created_at": 1781768225.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5.4-mini-2026-03-17",
  "object": "response",
  "output": [
    {
      "id": "msg_009f373e38174c09006a33a021751c81a38b6b39881ffd244d",
      "content": [
        {
          "annotations": [],
          "text": "Submit it through the **course platform**—that’s where homework submissions are handled.\n\nIf you mean the **Module 1 homework**, first make sure you’ve followed the instructions in the **GitHub homework page**:\n- https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/01-agentic-rag/homework.md\n\nThen use the **course management platform** to enter your answers before the deadline.\n\nIf you want, I can also point you to the exact submission page for your cohort.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assi

In [58]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price + 
    response.usage.output_tokens * output_price
)

cost

0.0011220000000000002

In [65]:
def llm(instructions, user_prompt, model='gpt-5.4-mini'):
    message_history = [
        {'role': 'system', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text


In [66]:
def rag(query, model='gpt-5.4-mini'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [67]:
rag('What will i learn?')

'You’ll learn about:\n\n- **RAG (Retrieval-Augmented Generation)**\n- **Vector search**\n- **Embeddings**\n- **Cosine similarity**\n\nIf you want, I can also explain what each of these means in simple terms.'